In [1]:
# Challenge

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets

from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display

import warnings
warnings.filterwarnings('ignore')

# Style des graphiques
plt.style.use('ggplot')
sns.set_style("whitegrid")

In [ ]:
# Charger le dataset
import pandas as pd

df = pd.read_csv(
    r"C:\Users\HP 840 G3\DAILYCHALLENGES\Week2\Day4\superstore_dataset.csv",
    encoding='latin1'
)

print(df.head())

# Aperçu
print("Shape du dataset :", df.shape)

print("\nColonnes :")
print(df.columns.tolist())

# Informations générales
df.info()

# Statistiques descriptives
df.describe()

# Valeurs manquantes
print("\nValeurs manquantes :")
print(df.isnull().sum())

In [ ]:
print("Nombre de doublons :", df.duplicated().sum())

# Suppression
df = df.drop_duplicates()

In [ ]:
print(df.isnull().sum())

# Exemple : remplacer les codes postaux manquants
if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)

In [ ]:
# Colonnes de dates
date_columns = ['Order Date', 'Ship Date']

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

# Vérification
print(df[date_columns].dtypes)

In [ ]:
# Marge bénéficiaire
df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100

# Année
df['Order Year'] = df['Order Date'].dt.year

# Mois
df['Order Month'] = df['Order Date'].dt.month

# Mois + année
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

# Aperçu
print(df[['Sales', 'Profit', 'Profit Margin',
          'Order Year', 'Order Month']].head())

In [19]:
monthly_sales = df.groupby(
    ['Order Month-Year', 'Category']
)['Sales'].sum().reset_index()

monthly_sales['Date'] = monthly_sales[
    'Order Month-Year'
].dt.to_timestamp()

In [ ]:
def plot_monthly_sales(category='All'):

    plt.figure(figsize=(12,6))

    if category == 'All':

        total_monthly = df.groupby(
            'Order Month-Year'
        )['Sales'].sum()

        plt.plot(
            total_monthly.index.to_timestamp(),
            total_monthly.values,
            marker='o',
            linewidth=2
        )

        plt.title(
            'Monthly Sales Trend - All Categories',
            fontsize=16,
            fontweight='bold'
        )

    else:

        category_data = monthly_sales[
            monthly_sales['Category'] == category
        ]

        plt.plot(
            category_data['Date'],
            category_data['Sales'],
            marker='o',
            linewidth=2
        )

        plt.title(
            f'Monthly Sales Trend - {category}',
            fontsize=16,
            fontweight='bold'
        )

    plt.xlabel('Date')
    plt.ylabel('Sales ($)')
    plt.xticks(rotation=45)

    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Widget interactif
categories = ['All'] + list(df['Category'].unique())

category_dropdown = Dropdown(
    options=categories,
    value='All',
    description='Category:'
)

interact(plot_monthly_sales,
         category=category_dropdown)

In [21]:
state_sales = df.groupby(
    'State'
)['Sales'].sum().sort_values(ascending=True)

In [ ]:
def plot_top_states(top_n=10):

    plt.figure(figsize=(12,6))

    top_states = state_sales.tail(top_n)

    bars = plt.barh(
        range(len(top_states)),
        top_states.values
    )

    plt.yticks(
        range(len(top_states)),
        top_states.index
    )

    plt.xlabel('Total Sales ($)')
    plt.ylabel('State')

    plt.title(
        f'Top {top_n} States by Sales',
        fontsize=16,
        fontweight='bold'
    )

    # Valeurs sur les barres
    for i, value in enumerate(top_states.values):

        plt.text(
            value,
            i,
            f'${value:,.0f}',
            va='center'
        )

    plt.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()

# Slider interactif
top_n_slider = IntSlider(
    min=5,
    max=25,
    value=10,
    description='Top N'
)

interact(plot_top_states,
         top_n=top_n_slider)

In [ ]:
product_profit = df.groupby(
    'Product Name'
)['Profit'].sum().sort_values(
    ascending=False
).head(10)

plt.figure(figsize=(12,8))

ax = sns.barplot(
    x=product_profit.values,
    y=product_profit.index,
    palette='viridis',
    orient='h'
)

plt.title(
    'Top 10 Most Profitable Products',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Total Profit ($)')
plt.ylabel('Product Name')

# Ajouter les valeurs
for i, profit in enumerate(product_profit.values):

    ax.text(
        profit,
        i,
        f'${profit:,.0f}',
        va='center'
    )

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14,8))

sns.scatterplot(
    data=df,
    x='Discount',
    y='Profit',
    hue='Category',
    alpha=0.6,
    s=50
)

# Courbe de tendance
sns.regplot(
    data=df,
    x='Discount',
    y='Profit',
    scatter=False,
    color='red'
)

plt.axhline(
    y=0,
    color='black',
    linestyle='--'
)

plt.title(
    'Discount vs Profit Analysis',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Discount')
plt.ylabel('Profit')

plt.tight_layout()
plt.show()

In [ ]:
high_discount = df[df['Discount'] > 0.2]

print("Transactions >20% remise :",
      len(high_discount))

print("Profit moyen :",
      high_discount['Profit'].mean())

print(
    "Pourcentage de pertes :",
    (high_discount['Profit'] < 0).mean() * 100
)

In [ ]:
print("=== COMPARAISON ===")

print("\nMATPLOTLIB")
print("- Contrôle détaillé")
print("- Widgets interactifs")
print("- Flexible")

print("\nSEABORN")
print("- Plus esthétique")
print("- Rapide pour statistiques")
print("- Très bon pour dashboards")

In [ ]:
def create_dashboard():

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2,
        figsize=(16,12)
    )

    # Sales trend
    monthly_total = df.groupby(
        'Order Month-Year'
    )['Sales'].sum()

    ax1.plot(
        monthly_total.index.to_timestamp(),
        monthly_total.values
    )

    ax1.set_title('Monthly Sales Trend')

    # Category sales
    category_sales = df.groupby(
        'Category'
    )['Sales'].sum()

    ax2.bar(
        category_sales.index,
        category_sales.values
    )

    ax2.set_title('Sales by Category')

    # Top states
    top_states = state_sales.tail(10)

    ax3.barh(
        top_states.index,
        top_states.values
    )

    ax3.set_title('Top States')

    # Discount vs profit
    for category in df['Category'].unique():

        cat_data = df[
            df['Category'] == category
        ]

        ax4.scatter(
            cat_data['Discount'],
            cat_data['Profit'],
            label=category,
            alpha=0.6
        )

    ax4.axhline(y=0,
                color='black',
                linestyle='--')

    ax4.legend()

    ax4.set_title('Discount vs Profit')

    plt.tight_layout()
    plt.show()

# Lancer le dashboard
create_dashboard()

In [ ]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()

profit_margin = (
    total_profit / total_sales
) * 100

print("=== EXECUTIVE SUMMARY ===")

print(f"Total Revenue: ${total_sales:,.0f}")

print(f"Total Profit: ${total_profit:,.0f}")

print(f"Profit Margin: {profit_margin:.1f}%")

# Top state
top_state = state_sales.index[-1]

print(f"Best State: {top_state}")

# Catégorie leader
top_category = df.groupby(
    'Category'
)['Sales'].sum().sort_values(
    ascending=False
).index[0]

print(f"Top Category: {top_category}")

# Produit le plus rentable
print(f"Most Profitable Product: {product_profit.index[0]}")